**Objectif : prédire la région d’une personne à partir de ses préférences musicales.** 

 13 genres musicaux (`Q5_1` à `Q5_13`) et la région (`REG13`)
 Une ligne correspond à une personne

In [22]:
import pandas as pd

In [23]:
filepath = "../hadopi-2020-musique-data.ods"
df = pd.read_excel(filepath, engine="odf", na_values=["#NUL!"])
df.shape

(2200, 261)

## Garder les colonnes utiles
On écarte les autres colonnes, comme l’âge, le sexe, et les autres informations

La cible est `REG13`

In [24]:
style = [f"Q5_{i}" for i in range(1, 14)]
colonnes_utiles = style + ["REG13"]

dataset = df[colonnes_utiles].copy()

# Convertir le code d’absence avant filtrage
dataset = dataset.replace("#NUL!", pd.NA)
dataset.head()

,Q5_1,Q5_2,Q5_3,Q5_4,Q5_5,Q5_6,Q5_7,Q5_8,Q5_9,Q5_10,Q5_11,Q5_12,Q5_13,REG13
0,Variété Française,0,0,0,Jazz,0,0,0,0,Soul,0,0,0,LanguedicRoussillonMidiP
1,0,Pop / Rock,0,0,Jazz,0,0,0,0,0,0,Musiques du monde,0,PaysDeLaLoire1
2,Variété Française,Pop / Rock,Rap / Hip-Hop,0,0,0,0,0,R&B,0,0,0,0,NordPasDeCalaisPicardie
3,Variété Française,0,0,Classique,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,Variété Française,0,0,0,Jazz,0,0,0,0,0,0,Musiques du monde,0,AuvergneRhoneAlpes


In [25]:
dataset.isna().sum()

Q5_1     214
Q5_2     214
Q5_3     214
Q5_4     214
Q5_5     214
Q5_6     214
Q5_7     214
Q5_8     214
Q5_9     214
Q5_10    214
Q5_11    214
Q5_12    214
Q5_13    214
REG13      0
dtype: int64

## On traite les valeurs manquantes
La ligne `replace("#NUL!", pd.NA)` ==> convertis les absents en vraies valeurs manquantes, même s’ils n’ont pas été reconnus à la lecture du fichier. `dropna()` les retirer

Un `0` reste une bonne réponse 

In [26]:
styles = dataset[style].isna().all(axis=1)
style_incomplets = dataset[style].isna().any(axis=1)

print("Personnes sans réponse musicale :", styles.sum())
print("style partiellement renseignés :", (style_incomplets & ~styles).sum())
print("Région manquante :", dataset["REG13"].isna().sum())

Personnes sans réponse musicale : 214
style partiellement renseignés : 0
Région manquante : 0


Les **214 personnes sans genre renseigné** sont exclues de cette première version sur 2200 

On ne transforme pas leurs absences en `0`, car cela inventerait des réponses.
Il reste donc 1 986 personnes

In [27]:
nombre_depart = len(dataset)
dataset = dataset.dropna(subset=colonnes_utiles).copy()
dataset = dataset.reset_index(drop=True)

print("Personnes au départ :", nombre_depart)
print("Personnes retirées :", nombre_depart - len(dataset))
print("Personnes conservées :", len(dataset))

Personnes au départ : 2200
Personnes retirées : 214
Personnes conservées : 1986


## Transformer les préférences en nombres
**1 = sélectionné, 0 = non sélectionné.**

On repart des données brutes pour le codage, 
Les zéros numériques ou textuels (`0`) sont reconnus et les espaces autour des réponses sont retirés.

Si une réponse est inconnue, sa colonne et sa valeur sont affichées => trouver la cause au lieu de remplacer une réponse lambda

Le remplacement de `#NUL!` est aussi fait ici, avant `dropna()`

In [28]:
# Repartir des données brutes, sans réutiliser un codage précédent.
profils_complets = df[colonnes_utiles].copy()
profils_complets = profils_complets.replace("#NUL!", pd.NA)
profils_complets = profils_complets.dropna(subset=colonnes_utiles).copy()
dataset_code = profils_complets.copy()

codage = {
    '0': 0,
    '1': 1,
    'Variété Française': 1,
    'Pop / Rock': 1,
    'Rap / Hip-Hop': 1,
    'Classique': 1,
    'Jazz': 1,
    'Dance': 1,
    'Electronique': 1,
    'Métal': 1,
    'R&B': 1,
    'Soul': 1,
    'Reggae': 1,
    'Musiques du monde': 1,
    'Autres genres': 1,
}

for colonne in style:
    reponses = profils_complets[colonne].astype("string").str.strip()
    inconnues = reponses[~reponses.isin(codage)]

    if not inconnues.empty:
        print("Réponses non reconnues dans", colonne)
        display(inconnues.value_counts(dropna=False))

    dataset_code[colonne] = reponses.map(codage)

In [29]:
# Une valeur non reconnue est affichée en haut
assert dataset_code[style].notna().all().all(), "Vérifier les réponses affichées au-dessus."

dataset_code[style] = dataset_code[style].astype(int)
dataset_code.head()

,Q5_1,Q5_2,Q5_3,Q5_4,Q5_5,Q5_6,Q5_7,Q5_8,Q5_9,Q5_10,Q5_11,Q5_12,Q5_13,REG13
0,1,0,0,0,1,0,0,0,0,1,0,0,0,LanguedicRoussillonMidiP
1,0,1,0,0,1,0,0,0,0,0,0,1,0,PaysDeLaLoire1
2,1,1,1,0,0,0,0,0,1,0,0,0,0,NordPasDeCalaisPicardie
3,1,0,0,1,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,AuvergneRhoneAlpes


## noms aux colonnes
Les noms des styles

In [30]:
noms_colonnes = {
    "Q5_1": "variete_francaise",
    "Q5_2": "pop_rock",
    "Q5_3": "rap_hip_hop",
    "Q5_4": "classique",
    "Q5_5": "jazz",
    "Q5_6": "dance",
    "Q5_7": "electronique",
    "Q5_8": "metal",
    "Q5_9": "rnb",
    "Q5_10": "soul",
    "Q5_11": "reggae",
    "Q5_12": "musiques_du_monde",
    "Q5_13": "autres_genres",
    "REG13": "region"
}

dataset = dataset_code.rename(columns=noms_colonnes).reset_index(drop=True)
dataset.head()

,variete_francaise,pop_rock,rap_hip_hop,classique,jazz,dance,electronique,metal,rnb,soul,reggae,musiques_du_monde,autres_genres,region
0,1,0,0,0,1,0,0,0,0,1,0,0,0,LanguedicRoussillonMidiP
1,0,1,0,0,1,0,0,0,0,0,0,1,0,PaysDeLaLoire1
2,1,1,1,0,0,0,0,0,1,0,0,0,0,NordPasDeCalaisPicardie
3,1,0,0,1,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,AuvergneRhoneAlpes


## Dataset final

In [ ]:
controle = pd.DataFrame({
    "Type": dataset.dtypes,
    "Valeurs manquantes": dataset.isna().sum(),
    "Valeurs distinctes": dataset.nunique()
})

display(controle)

assert dataset.isna().sum().sum() == 0, "Il reste des valeurs manquantes."
assert dataset.shape[1] == 14, "On attend 13 préférences et une région."

print("Dimensions finales :", dataset.shape)
print("Nombre de régions :", dataset["region"].nunique())

,Type,Valeurs manquantes,Valeurs distinctes
variete_francaise,int64,0,2
pop_rock,int64,0,2
rap_hip_hop,int64,0,2
classique,int64,0,2
jazz,int64,0,2
dance,int64,0,2
electronique,int64,0,2
metal,int64,0,2
rnb,int64,0,2
soul,int64,0,2


Dimensions finales : (1986, 14)
Nombre de régions : 12


In [32]:
repartition_regions = pd.DataFrame({
    "Personnes": dataset["region"].value_counts(),
    "Part (%)": (dataset["region"].value_counts(normalize=True) * 100).round(1)
})

display(repartition_regions)

,Personnes,Part (%)
region,,
IleDeFrance1,404,20.3
AuvergneRhoneAlpes,237,11.9
AlsaceChampagneArdenneLo,200,10.1
AquitaineLimousinPoitouC,194,9.8
NordPasDeCalaisPicardie,188,9.5
LanguedicRoussillonMidiP,161,8.1
ProvenceAlpesCoteDAzur,129,6.5
PaysDeLaLoire1,125,6.3
Bretagne1,97,4.9


Le fichier contient 12 régions. 
Pzr contre, les classes n’ont pas toutes le même effecti

In [35]:
print("Lignes identiques sur les préférences et la région :", dataset.duplicated().sum())

Lignes identiques sur les préférences et la région : 638


Plusieurs personnes peuvent aimer les mêmes style et vivre dans la même région, les 638 n'est pas u doublons

## Préparer les entrées et la cible
`X` contient les 13 style de musique

`y`  la région à prédire. 

(La région n’entre pas dans `X`, pour éviter de donner la réponse au modèle)

In [36]:
X = dataset.drop(columns="region")
y = dataset["region"]

print("Préférences X :", X.shape)
print("Région y :", y.shape)

Préférences X : (1986, 13)
Région y : (1986,)


##  nouveau dataset
Le CSV contient les préférences en `0/1` et la cible `region`

In [37]:
dataset.to_csv("dataset_musique_region.csv", index=False, encoding="utf-8")

In [38]:
# Vérifier que le fichier enregistré contient exactement les mêmes données.
verification = pd.read_csv("dataset_musique_region.csv")
assert verification.equals(dataset), "Le fichier enregistré diffère du dataset."

print("CSV enregistré :", verification.shape)
verification.head()

CSV enregistré : (1986, 14)


,variete_francaise,pop_rock,rap_hip_hop,classique,jazz,dance,electronique,metal,rnb,soul,reggae,musiques_du_monde,autres_genres,region
0,1,0,0,0,1,0,0,0,0,1,0,0,0,LanguedicRoussillonMidiP
1,0,1,0,0,1,0,0,0,0,0,0,1,0,PaysDeLaLoire1
2,1,1,1,0,0,0,0,0,1,0,0,0,0,NordPasDeCalaisPicardie
3,1,0,0,1,0,0,0,0,0,0,0,0,0,ProvenceAlpesCoteDAzur
4,1,0,0,0,1,0,0,0,0,0,0,1,0,AuvergneRhoneAlpes


**Résultat : 1 986 personnes, 13 préférences musicales et 1 région cible.**